# Universal Hyperbolic Geometry: An Interactive Walkthrough

This notebook demonstrates the key principles of UHG using the `uhg` Python library.

UHG takes a **pure algebraic/projective approach** to hyperbolic geometry — no tangent space mappings,
no exponential or logarithmic maps, just direct computation in projective coordinates.

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from uhg.projective import ProjectiveUHG
from uhg.core import UHGCore

print(f"PyTorch version: {torch.__version__}")
import uhg
print(f"UHG version: {uhg.__version__}")

## 1. What is UHG?

Universal Hyperbolic Geometry (UHG) is an approach to hyperbolic geometry that operates
entirely within **projective coordinates**. Here are its defining features:

| Feature | Description |
|---------|-------------|
| **Projective coordinates** | Points are represented as homogeneous triples $[x : y : z]$ in $\mathbb{RP}^2$. |
| **The null cone** | The quadratic form $x^2 + y^2 - z^2 = 0$ defines the boundary. Points with $x^2 + y^2 - z^2 < 0$ are *hyperbolic* (inside the disk). |
| **Universality** | The theory works over *any* ordered field, not just the reals — hence "Universal". |
| **No curvature parameter** | Unlike the Poincaré or Lorentz models, UHG needs no curvature constant $c$. |

The **Minkowski inner product** $\langle a, b \rangle_M = x_1 x_2 + y_1 y_2 - z_1 z_2$ replaces
the Euclidean dot product. Points with negative Minkowski norm live inside the hyperbolic disk;
points with zero norm lie on the null cone (boundary).

In [ ]:
uhg_proj = ProjectiveUHG()

# --- Null cone boundary (the Beltrami-Klein disk edge) ---
theta = np.linspace(0, 2 * np.pi, 200)
boundary_x = np.cos(theta)
boundary_y = np.sin(theta)

# --- Sample points inside the disk ---
points = [
    ([0.3, 0.2, 1.0],  'P1', 'Inside (hyperbolic)'),
    ([-0.4, 0.1, 1.0], 'P2', 'Inside (hyperbolic)'),
    ([0.1, -0.5, 1.0], 'P3', 'Inside (hyperbolic)'),
    ([0.0, 0.0, 1.0],  'O',  'Origin'),
]
colors = ['#e74c3c', '#2980b9', '#27ae60', '#8e44ad']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=boundary_x.tolist(), y=boundary_y.tolist(),
    mode='lines', name='Null Cone (boundary)',
    line=dict(color='black', width=2.5),
    hoverinfo='name'
))

# Shade the hyperbolic interior
fig.add_trace(go.Scatter(
    x=boundary_x.tolist(), y=boundary_y.tolist(),
    fill='toself', fillcolor='rgba(173,216,230,0.15)',
    line=dict(width=0), showlegend=False, hoverinfo='skip'
))

for (coords, label, desc), color in zip(points, colors):
    pt = torch.tensor(coords)
    mink_norm = uhg_proj.inner_product(pt, pt).item()
    is_null = uhg_proj.is_null_point(pt).item()
    px, py = coords[0] / coords[2], coords[1] / coords[2]
    fig.add_trace(go.Scatter(
        x=[px], y=[py],
        mode='markers+text',
        text=[label],
        textposition='top right',
        textfont=dict(size=13, color=color),
        marker=dict(size=11, color=color, line=dict(width=1, color='white')),
        name=f'{label}: [{coords[0]}:{coords[1]}:{coords[2]}]',
        hovertext=f'{desc}<br>Minkowski norm = {mink_norm:.4f}<br>Null = {is_null}',
        hoverinfo='text+name',
    ))

fig.update_layout(
    title=dict(text='The Beltrami-Klein Disk Model', font=dict(size=18)),
    xaxis=dict(title='x', range=[-1.35, 1.35], zeroline=True, zerolinecolor='#ccc',
               scaleanchor='y', scaleratio=1),
    yaxis=dict(title='y', range=[-1.35, 1.35], zeroline=True, zerolinecolor='#ccc'),
    width=620, height=620,
    template='plotly_white',
    annotations=[dict(x=0.85, y=-0.1, text='x²+y²−z² < 0<br>(hyperbolic)',
                       showarrow=False, font=dict(size=11, color='#555'))],
)
fig.show()

## 2. Join and Meet: The Fundamental Operations

In projective geometry, the two fundamental incidence operations are:

- **Join** $P \vee Q$: computes the *line* through two *points* using the cross product (wedge product).
- **Meet** $L \wedge M$: computes the *point* of intersection of two *lines* — the same cross-product formula, applied to line coordinates.

This duality — the same algebraic operation works for both — is one of the elegant features of projective geometry.

**Incidence rule:** A point $P = [x:y:z]$ lies on a line $L = (a:b:c)$ if and only if $ax + by + cz = 0$.

In [ ]:
uhg_proj = ProjectiveUHG()

p1 = torch.tensor([0.3, 0.2, 1.0])
p2 = torch.tensor([-0.4, 0.1, 1.0])
p3 = torch.tensor([0.1, -0.5, 1.0])
p4 = torch.tensor([-0.2, 0.4, 1.0])

# --- Join: line through two points ---
L1 = uhg_proj.join(p1, p2)
L2 = uhg_proj.join(p3, p4)
print("Join (line through two points):")
print(f"  L1 = join(P1, P2) = [{L1[0]:.4f} : {L1[1]:.4f} : {L1[2]:.4f}]")
print(f"  L2 = join(P3, P4) = [{L2[0]:.4f} : {L2[1]:.4f} : {L2[2]:.4f}]")

# --- Incidence check ---
print(f"\nIncidence checks (should be ≈ 0):")
p1n = uhg_proj.normalize_points(p1)
p2n = uhg_proj.normalize_points(p2)
print(f"  P1_norm · L1 = {torch.dot(p1n, L1).item():.2e}")
print(f"  P2_norm · L1 = {torch.dot(p2n, L1).item():.2e}")

# --- Meet: intersection of two lines (cross product by duality) ---
intersection = torch.cross(L1, L2, dim=-1)
if intersection[2].abs() > 1e-8:
    int_affine = intersection / intersection[2]
    print(f"\nMeet (intersection of L1 ∩ L2):")
    print(f"  I = [{int_affine[0]:.4f} : {int_affine[1]:.4f} : {int_affine[2]:.4f}]")
    print(f"  I · L1 = {torch.dot(intersection, L1).item():.2e} (≈ 0)")
    print(f"  I · L2 = {torch.dot(intersection, L2).item():.2e} (≈ 0)")
else:
    print("\nLines are parallel (meet at infinity).")

# --- Visualization ---
theta = np.linspace(0, 2 * np.pi, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.cos(theta).tolist(), y=np.sin(theta).tolist(),
                          mode='lines', name='Null Cone', line=dict(color='black', width=2)))

for pt, nm, clr in [(p1,'P1','#e74c3c'), (p2,'P2','#2980b9'),
                     (p3,'P3','#27ae60'), (p4,'P4','#e67e22')]:
    fig.add_trace(go.Scatter(
        x=[pt[0].item()/pt[2].item()], y=[pt[1].item()/pt[2].item()],
        mode='markers+text', text=[nm], textposition='top right',
        marker=dict(size=10, color=clr), name=nm))

def line_segment_in_disk(line_coeffs, n=300):
    """Clip a projective line to the interior of the unit disk."""
    a, b, c = [v.item() for v in line_coeffs]
    t = np.linspace(-2, 2, n)
    if abs(b) > abs(a):
        lx = t
        ly = -(a * t + c) / b
    else:
        ly = t
        lx = -(b * t + c) / a
    mask = (lx**2 + ly**2) < 0.999
    return lx[mask], ly[mask]

for ln, nm, clr in [(L1, 'L1 = join(P1,P2)', '#9b59b6'),
                     (L2, 'L2 = join(P3,P4)', '#e67e22')]:
    lx, ly = line_segment_in_disk(ln)
    if len(lx) > 0:
        fig.add_trace(go.Scatter(x=lx.tolist(), y=ly.tolist(), mode='lines',
                                  name=nm, line=dict(dash='dash', width=2, color=clr)))

if intersection[2].abs() > 1e-8:
    ix = int_affine[0].item()
    iy = int_affine[1].item()
    if ix**2 + iy**2 < 1:
        fig.add_trace(go.Scatter(x=[ix], y=[iy], mode='markers',
            marker=dict(size=14, color='black', symbol='x-thin-open', line=dict(width=3)),
            name='L1 ∩ L2'))

fig.update_layout(
    title='Join and Meet in the Beltrami-Klein Disk',
    xaxis=dict(range=[-1.35, 1.35], scaleanchor='y', scaleratio=1, zeroline=False),
    yaxis=dict(range=[-1.35, 1.35], zeroline=False),
    width=620, height=620, template='plotly_white',
)
fig.show()

## 3. Quadrance: The UHG Distance Measure

Instead of a distance *metric*, UHG uses **quadrance** — a purely algebraic quantity:

$$Q(a, b) = 1 - \frac{\langle a, b \rangle_M^{\,2}}{\langle a, a \rangle_M \; \langle b, b \rangle_M}$$

where $\langle a, b \rangle_M = x_1 x_2 + y_1 y_2 - z_1 z_2$ is the **Minkowski inner product**.

Key properties:
- Quadrance is **not** a distance metric — it can take values in $[0, 1]$ for hyperbolic points.
- It requires **no logarithmic map** — it is computed directly from projective coordinates.
- Identical points have $Q = 0$; points near the boundary approach $Q = 1$.
- The formula is rational (no square roots or transcendental functions).

In [ ]:
uhg_proj = ProjectiveUHG()

point_pairs = [
    (torch.tensor([0.0, 0.0, 1.0]),  torch.tensor([0.1, 0.05, 1.0]),  'Origin → nearby'),
    (torch.tensor([0.3, 0.2, 1.0]),  torch.tensor([-0.4, 0.1, 1.0]),  'P1 → P2'),
    (torch.tensor([0.0, 0.0, 1.0]),  torch.tensor([0.7, 0.0, 1.0]),   'Origin → far'),
    (torch.tensor([0.5, 0.5, 1.0]),  torch.tensor([-0.5, -0.5, 1.0]), 'Opposite sides'),
]

print("Quadrance between point pairs")
print("=" * 55)
q_vals = []
for a, b, label in point_pairs:
    try:
        q = uhg_proj.quadrance(a, b)
        q_vals.append(q.item())
        mink_a = uhg_proj.inner_product(a, a).item()
        mink_b = uhg_proj.inner_product(b, b).item()
        print(f"  {label:22s}  Q = {q.item():.6f}   "
              f"(‖a‖_M={mink_a:+.3f}, ‖b‖_M={mink_b:+.3f})")
    except ValueError as e:
        q_vals.append(None)
        print(f"  {label:22s}  Error: {e}")

# --- Visualization: disk with color-coded quadrance segments ---
theta = np.linspace(0, 2 * np.pi, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.cos(theta).tolist(), y=np.sin(theta).tolist(),
                          mode='lines', name='Null Cone', line=dict(color='black', width=2)))

cmap = ['#3498db', '#e74c3c', '#f39c12', '#8e44ad']
for i, (a, b, label) in enumerate(point_pairs):
    ax_v, ay_v = a[0].item() / a[2].item(), a[1].item() / a[2].item()
    bx_v, by_v = b[0].item() / b[2].item(), b[1].item() / b[2].item()
    q_str = f'Q={q_vals[i]:.4f}' if q_vals[i] is not None else 'undef'
    fig.add_trace(go.Scatter(
        x=[ax_v, bx_v], y=[ay_v, by_v],
        mode='markers+lines+text',
        text=[label.split('→')[0].strip(), q_str],
        textposition=['top left', 'bottom right'],
        textfont=dict(size=11, color=cmap[i]),
        marker=dict(size=9, color=cmap[i]),
        line=dict(color=cmap[i], width=2, dash='dot'),
        name=f'{label}  ({q_str})',
    ))

fig.update_layout(
    title='Quadrance Between Point Pairs',
    xaxis=dict(range=[-1.35, 1.35], scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[-1.35, 1.35]),
    width=620, height=620, template='plotly_white',
)
fig.show()

## 4. Cross-Ratio: The Fundamental Invariant

The **cross-ratio** of four points is the single most important invariant in projective
(and therefore hyperbolic) geometry:

$$\text{CR}(A, B; C, D) \;=\; \frac{\det(A, C)\;\det(B, D)}{\det(A, D)\;\det(B, C)}$$

where $\det(P, Q)$ is the $2 \times 2$ determinant of the first two coordinates.

**Key fact:** The cross-ratio is *preserved* under every projective transformation $T$:
$$\text{CR}(A, B; C, D) = \text{CR}(TA, TB; TC, TD)$$

This is *why* projective geometry works for hyperbolic space — the cross-ratio encodes
all the geometric information, and projective maps are exactly those maps that preserve it.
All UHG neural-network layers are designed around this invariance.

In [ ]:
uhg_proj = ProjectiveUHG()

# Four points inside the disk
A = torch.tensor([0.3,  0.2, 1.0])
B = torch.tensor([-0.4, 0.1, 1.0])
C = torch.tensor([0.1, -0.5, 1.0])
D = torch.tensor([-0.2, 0.4, 1.0])

cr_before = uhg_proj.cross_ratio(A, B, C, D)
print(f"Cross-ratio CR(A,B;C,D) = {cr_before.item():.6f}")

# Apply an arbitrary invertible projective transformation (3×3 matrix)
T = torch.tensor([
    [ 2.0,  0.5,  0.1],
    [ 0.3,  1.5, -0.2],
    [ 0.1, -0.1,  1.0],
])
print(f"\nTransformation matrix T (det = {torch.linalg.det(T).item():.3f}):")
print(T.numpy())

At = T @ A
Bt = T @ B
Ct = T @ C
Dt = T @ D

cr_after = uhg_proj.cross_ratio(At, Bt, Ct, Dt)
print(f"\nCR before transformation: {cr_before.item():.6f}")
print(f"CR after  transformation: {cr_after.item():.6f}")
print(f"Preserved: {torch.allclose(cr_before, cr_after, rtol=1e-4)}")

# Run many random transformations to confirm
print("\nStress test — 100 random projective transformations:")
num_preserved = 0
for _ in range(100):
    M = torch.randn(3, 3)
    while torch.linalg.det(M).abs() < 0.1:
        M = torch.randn(3, 3)
    cr_t = uhg_proj.cross_ratio(M @ A, M @ B, M @ C, M @ D)
    if torch.allclose(cr_before, cr_t, rtol=1e-4):
        num_preserved += 1
print(f"  Preserved in {num_preserved}/100 trials")

## 5. Why No Tangent Space?

Most hyperbolic geometry libraries follow this pipeline:

```
Hyperbolic point  ──log map──▶  Tangent vector (Euclidean)  ──operation──▶  Result  ──exp map──▶  Hyperbolic point
```

**Problems with the tangent-space approach:**

1. **Accumulated numerical error** — each log/exp round-trip introduces floating-point drift.
2. **Curvature-dependent** — the maps depend on a curvature parameter $c$ that must be tuned.
3. **Large-distance breakdown** — for points far apart, the tangent-space approximation degrades.

**The UHG approach:**

```
Projective point  ──algebraic operation──▶  Projective result
```

- All operations are polynomial/rational in the coordinates.
- No transcendental functions (no `exp`, `log`, `cosh`, `sinh`).
- Exact up to floating-point representation.
- Works uniformly regardless of point separation.

In [ ]:
uhg_proj = ProjectiveUHG()

def normalize_to_hyperboloid(p):
    """Project a point onto the hyperboloid x²+y²-z² = -1 (Lorentz model)."""
    spatial = p[:2]
    z = torch.sqrt(1.0 + torch.sum(spatial ** 2))
    return torch.cat([spatial, z.unsqueeze(0)])

def lorentz_distance_sq(a, b):
    """Tangent-space approach: normalize, compute acosh-based distance, square it.
    Requires the transcendental function acosh."""
    a_h = normalize_to_hyperboloid(a)
    b_h = normalize_to_hyperboloid(b)
    # Minkowski inner product
    inner = a_h[0]*b_h[0] + a_h[1]*b_h[1] - a_h[2]*b_h[2]
    # d = acosh(-inner), inner <= -1 for hyperboloid points
    arg = torch.clamp(-inner, min=1.0)
    d = torch.acosh(arg)
    return d ** 2

def uhg_quadrance(a, b):
    """Direct UHG quadrance — pure rational algebra, no transcendental functions."""
    return uhg_proj.quadrance(a, b)

print("Tangent-space distance² vs UHG quadrance")
print("=" * 60)
print("  (Different measures, but both quantify 'separation')")
print()

test_pairs = [
    (torch.tensor([0.01, 0.01, 1.0]),  torch.tensor([0.02, 0.015, 1.0]),  'Very close'),
    (torch.tensor([0.3, 0.2, 1.0]),    torch.tensor([-0.4, 0.1, 1.0]),    'Moderate'),
    (torch.tensor([0.1, 0.0, 1.0]),    torch.tensor([0.85, 0.0, 1.0]),    'Far apart'),
]

print(f"  {'Pair':15s}  {'UHG Q (rational)':>18s}  {'Lorentz d² (acosh)':>20s}")
print(f"  {'-'*15}  {'-'*18}  {'-'*20}")
for a, b, label in test_pairs:
    q_uhg = uhg_quadrance(a, b)
    try:
        d_sq = lorentz_distance_sq(a, b)
        print(f"  {label:15s}  {q_uhg.item():18.8f}  {d_sq.item():20.8f}")
    except Exception as e:
        print(f"  {label:15s}  {q_uhg.item():18.8f}  {'FAILED':>20s}")

print("\nKey insight: UHG quadrance uses only +, −, ×, ÷ on coordinates.")
print("The Lorentz distance requires acosh (transcendental), which introduces")
print("floating-point error that compounds over successive operations.")

# Demonstrate numerical stability under repeated midpoint computation
print("\n--- Numerical stability: 10 successive midpoint steps ---")
a = torch.tensor([0.3, 0.2, 1.0])
b = torch.tensor([-0.4, 0.1, 1.0])
q_original = uhg_proj.quadrance(a, b).item()

current = a.clone()
steps_taken = 0
for step in range(10):
    m1, m2 = uhg_proj.midpoints(current, b)
    if m1 is not None:
        current = m1
        steps_taken += 1
    else:
        break

q_after = uhg_proj.quadrance(current, b).item()
print(f"  Original  Q(a, b)       = {q_original:.8f}")
print(f"  After {steps_taken} midpoint steps  Q(m, b) = {q_after:.8f}")
print(f"  (Successive midpoints converge toward b — all computed algebraically)")

## 6. Spread and Duality

Just as **quadrance** measures separation between *points*, **spread** measures the
angular separation between *lines*:

$$S(L_1, L_2) = 1 - \frac{(L_1 \cdot L_2)^2}{(L_1 \cdot L_1)(L_2 \cdot L_2)}$$

This formula is identical in structure to quadrance — reflecting the **point-line duality**
of projective geometry. In UHG, every theorem about points and quadrances has a dual theorem
about lines and spreads.

In [ ]:
uhg_proj = ProjectiveUHG()

# Three points forming a triangle
A = torch.tensor([0.3, 0.2, 1.0])
B = torch.tensor([-0.3, 0.25, 1.0])
C = torch.tensor([0.0, -0.4, 1.0])

# Sides of the triangle (lines)
L_AB = uhg_proj.join(A, B)
L_BC = uhg_proj.join(B, C)
L_CA = uhg_proj.join(C, A)

# Quadrances (side "lengths")
q_AB = uhg_proj.quadrance(A, B)
q_BC = uhg_proj.quadrance(B, C)
q_CA = uhg_proj.quadrance(C, A)

print("Triangle in hyperbolic space")
print("=" * 45)
print(f"  Quadrance(A,B) = {q_AB.item():.6f}")
print(f"  Quadrance(B,C) = {q_BC.item():.6f}")
print(f"  Quadrance(C,A) = {q_CA.item():.6f}")

# Spreads (vertex "angles")
try:
    S_A = uhg_proj.spread(L_CA, L_AB)
    S_B = uhg_proj.spread(L_AB, L_BC)
    S_C = uhg_proj.spread(L_BC, L_CA)
    print(f"\n  Spread at A = {S_A.item():.6f}")
    print(f"  Spread at B = {S_B.item():.6f}")
    print(f"  Spread at C = {S_C.item():.6f}")
except ValueError as e:
    print(f"\n  Spread computation: {e}")

# Visualize the triangle
theta = np.linspace(0, 2 * np.pi, 200)
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.cos(theta).tolist(), y=np.sin(theta).tolist(),
                          mode='lines', name='Null Cone', line=dict(color='black', width=2)))

verts = [(A, 'A', '#e74c3c'), (B, 'B', '#2980b9'), (C, 'C', '#27ae60')]
for pt, nm, clr in verts:
    fig.add_trace(go.Scatter(
        x=[pt[0].item()/pt[2].item()], y=[pt[1].item()/pt[2].item()],
        mode='markers+text', text=[nm], textposition='top right',
        marker=dict(size=10, color=clr), name=nm))

sides = [
    (A, B, f'AB (Q={q_AB.item():.3f})', '#e74c3c'),
    (B, C, f'BC (Q={q_BC.item():.3f})', '#2980b9'),
    (C, A, f'CA (Q={q_CA.item():.3f})', '#27ae60'),
]
for p, q_pt, nm, clr in sides:
    fig.add_trace(go.Scatter(
        x=[p[0].item()/p[2].item(), q_pt[0].item()/q_pt[2].item()],
        y=[p[1].item()/p[2].item(), q_pt[1].item()/q_pt[2].item()],
        mode='lines', name=nm, line=dict(color=clr, width=2)))

fig.update_layout(
    title='Hyperbolic Triangle — Quadrance and Spread',
    xaxis=dict(range=[-1.35, 1.35], scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[-1.35, 1.35]),
    width=620, height=620, template='plotly_white',
)
fig.show()

## 7. UHG in Machine Learning

The UHG library provides neural-network layers that operate **directly in projective/hyperbolic space**:

| Layer | Description |
|-------|-------------|
| `UHGLayer` | Base class: projective transforms via orthogonal spatial blocks, Minkowski-norm normalization. |
| `ProjectiveSAGEConv` | GraphSAGE convolution using cross-ratio–weighted message passing. |
| `ProjectiveLinear` | Fully connected layer with projective weight matrices. |

All transformations use **projective matrices** — no `exp` / `log` maps appear anywhere
in the computational graph. The key invariant preserved through every layer is the **cross-ratio**.

Below we visualize what hyperbolic embeddings look like on the **hyperboloid model**
(the surface $z = \sqrt{1 + x^2 + y^2}$), which is the native space of UHG computations.

In [ ]:
uhg_proj = ProjectiveUHG()

# --- Generate the hyperboloid surface z = sqrt(1 + x² + y²) ---
grid = np.linspace(-1.2, 1.2, 60)
X, Y = np.meshgrid(grid, grid)
Z = np.sqrt(1.0 + X**2 + Y**2)

# --- Simulate embeddings: random points on the hyperboloid ---
np.random.seed(42)
n_points = 40
n_classes = 3
class_labels = np.random.randint(0, n_classes, n_points)
class_colors = ['#e74c3c', '#2980b9', '#27ae60']
class_names = ['Class A', 'Class B', 'Class C']

# Generate spatial coords clustered by class
centers = [np.array([0.4, 0.3]), np.array([-0.5, 0.1]), np.array([0.0, -0.6])]
emb_x, emb_y, emb_z = [], [], []
quadrances_within = []

for i in range(n_points):
    cx, cy = centers[class_labels[i]]
    sx = cx + np.random.randn() * 0.15
    sy = cy + np.random.randn() * 0.15
    sz = np.sqrt(1.0 + sx**2 + sy**2)
    emb_x.append(sx)
    emb_y.append(sy)
    emb_z.append(sz)

# --- 3D hyperboloid visualization ---
fig = go.Figure()

fig.add_trace(go.Surface(
    x=X, y=Y, z=Z,
    colorscale=[[0, 'rgba(200,220,240,0.3)'], [1, 'rgba(150,190,230,0.3)']],
    showscale=False, name='Hyperboloid',
    hoverinfo='skip', opacity=0.35,
))

for c in range(n_classes):
    mask = class_labels == c
    fig.add_trace(go.Scatter3d(
        x=np.array(emb_x)[mask].tolist(),
        y=np.array(emb_y)[mask].tolist(),
        z=np.array(emb_z)[mask].tolist(),
        mode='markers',
        marker=dict(size=5, color=class_colors[c], opacity=0.9,
                    line=dict(width=0.5, color='white')),
        name=class_names[c],
        hovertext=[f'({emb_x[i]:.2f}, {emb_y[i]:.2f}, {emb_z[i]:.2f})'
                   for i in range(n_points) if class_labels[i] == c],
    ))

# Draw the null cone ring at z=1
cone_theta = np.linspace(0, 2 * np.pi, 100)
fig.add_trace(go.Scatter3d(
    x=np.cos(cone_theta).tolist(),
    y=np.sin(cone_theta).tolist(),
    z=(np.ones_like(cone_theta) * np.sqrt(2.0)).tolist(),
    mode='lines', line=dict(color='black', width=3),
    name='Null Cone (z=√2)',
))

fig.update_layout(
    title=dict(text='Hyperbolic Embeddings on the Hyperboloid', font=dict(size=16)),
    scene=dict(
        xaxis_title='x', yaxis_title='y', zaxis_title='z',
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.0)),
        aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.8),
    ),
    width=750, height=650, template='plotly_white',
)
fig.show()

# Compute some inter-class vs intra-class quadrances
print("\nSample quadrances between embeddings:")
for i in range(min(5, n_points)):
    for j in range(i + 1, min(6, n_points)):
        pi = torch.tensor([emb_x[i], emb_y[i], emb_z[i]])
        pj = torch.tensor([emb_x[j], emb_y[j], emb_z[j]])
        try:
            q = uhg_proj.quadrance(pi, pj).item()
            same_class = '✓' if class_labels[i] == class_labels[j] else '✗'
            print(f"  Q({i},{j}) = {q:.4f}  "
              f"[{class_names[class_labels[i]]} {same_class} {class_names[class_labels[j]]}]")
        except ValueError:
            pass

## 8. Putting It Together: The UHG Toolkit

Here is a summary of the core operations available in the `uhg` library and the principles behind them:

| Operation | Method | Formula / Principle |
|-----------|--------|--------------------|
| **Join** | `uhg.join(P, Q)` | Cross product of point coordinates → line |
| **Meet** | `torch.cross(L1, L2)` | Cross product of line coordinates → point (duality) |
| **Quadrance** | `uhg.quadrance(P, Q)` | $1 - \langle P,Q\rangle_M^2 / (\langle P,P\rangle_M \langle Q,Q\rangle_M)$ |
| **Spread** | `uhg.spread(L1, L2)` | Same formula applied to lines |
| **Cross-ratio** | `uhg.cross_ratio(A,B,C,D)` | Preserved under all projective transformations |
| **Midpoints** | `uhg.midpoints(P, Q)` | Algebraic construction (Theorem 54) |
| **Transform** | `uhg.transform(P, M)` | Projective matrix multiplication |
| **Distance** | `uhg.distance(P, Q)` | $\text{arccosh}(-\langle P,Q\rangle_M)$ (for comparison) |

### Design philosophy

1. **No tangent space** — every operation works directly in projective coordinates.
2. **Cross-ratio preservation** — the fundamental invariant, enforced by construction.
3. **Algebraic** — formulas are rational (no transcendentals), enabling exact computation.
4. **Universal** — the same formulas work over any ordered field.
5. **ML-ready** — `ProjectiveSAGEConv` and other layers plug into PyTorch's autograd.

For more, explore:
- `uhg.nn.layers` — neural-network layers
- `uhg.nn.models` — full model architectures (HGNN, HGT, hierarchical)
- `uhg.datasets` — graph datasets with hyperbolic structure
- `uhg.optim` — UHG-aware optimizers